# Whisper (варианты encoder) + SVM

**Модель:** Whisper encoder (frozen) → mean+std+max → SVM RBF

`MODEL_FILTER` позволяет запустить один вариант; `None` — запустить все.

| Вариант | MODEL_ID |
|---------|----------|
| small | openai/whisper-small |
| medium | openai/whisper-medium |
| large | openai/whisper-large |
| dysarthria | qymyz/whisper-russian-dysarthria |

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import mlflow
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from transformers import WhisperProcessor, WhisperModel

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils
from shared.evaluate import find_optimal_threshold, evaluate, evaluate_cv_folds, print_cv_summary
from shared.data_utils import build_feature_matrix, get_cv_folds, get_durations, load_audio
from shared.results_utils import save_result_csv
from shared.mlflow_utils import start_run, log_cv_fold

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda:0


In [2]:
# Конфигурации моделей
WHISPER_MODELS = [
    {
        "model_id":    "openai/whisper-small",
        "exp_id":      "exp_whisper_svm",
        "exp_name":    "Whisper-small encoder + SVM",
        "model_label": "Whisper-small (frozen) + SVM RBF",
    },
    {
        "model_id":    "openai/whisper-medium",
        "exp_id":      "exp_whisper_medium_svm",
        "exp_name":    "Whisper-medium encoder + SVM",
        "model_label": "Whisper-medium (frozen) + SVM RBF",
    },
    {
        "model_id":    "openai/whisper-large",
        "exp_id":      "exp_whisper_large_svm",
        "exp_name":    "Whisper-large encoder + SVM",
        "model_label": "Whisper-large (frozen) + SVM RBF",
    },
    {
        "model_id":    "qymyz/whisper-russian-dysarthria",
        "exp_id":      "exp_whisper_dysarthria_svm",
        "exp_name":    "Whisper (Russian dysarthria finetune) + SVM",
        "model_label": "qymyz/whisper-russian-dysarthria (frozen) + SVM RBF",
        "notes":       "Whisper fine-tuned on Russian dysarthria speech",
    },
]

# Для запуска одного варианта: MODEL_FILTER = "openai/whisper-small"
MODEL_FILTER = None  # None — запустить все

print(f"Device: {DEVICE}")

Device: cuda:0


In [3]:
def make_whisper_extractor(model_id, DEVICE):
    """Загружает Whisper-модель и возвращает (extract_fn, model, embed_dim).
    Вызывающий код должен явно удалить model после извлечения признаков.
    """
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperModel.from_pretrained(model_id).to(DEVICE)
    model.eval()
    embed_dim = model.config.d_model * 3  # mean + std + max

    def extract_fn(path):
        y, _ = load_audio(path, sr=16000)
        feats = processor.feature_extractor(y, sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            hs = model.encoder(
                input_features=feats.input_features.to(DEVICE)
            ).last_hidden_state[0].cpu().numpy()
        return np.concatenate([hs.mean(0), hs.std(0), hs.max(0)]).astype(np.float32)

    return extract_fn, model, embed_dim

In [4]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_holdout_split()
print(f"Train+Val: {len(paths_trainval)}  |  Test: {len(paths_test)}")

# Индекс путей для быстрого поиска при нарезке фолдов
path_to_idx = {p: i for i, p in enumerate(paths_trainval)}

Train+Val: 2356  |  Test: 416


In [5]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.05, 0.1],
}


def run_whisper_experiment(cfg, result_csv_exists):
    """Запускает Этап 1 (извлечение эмбеддингов) + Этап 2 (5-fold CV).

    Возвращает dict с данными для последующей оценки на тесте.
    """
    model_id    = cfg["model_id"]
    exp_id      = cfg["exp_id"]
    extra_notes = cfg.get("notes", "")

    print(f"\n{'='*60}")
    print(f"Модель: {model_id}")
    print(f"{'='*60}")

    # --- Этап 1: извлечение эмбеддингов (GPU) ---
    extract_fn, whisper_model, embed_dim = make_whisper_extractor(model_id, DEVICE)

    print("Извлечение признаков (train+val)...")
    X_embeds_trainval = build_feature_matrix(paths_trainval, extract_fn, n_jobs=1)
    print("Извлечение признаков (test)...")
    X_embeds_test     = build_feature_matrix(paths_test,     extract_fn, n_jobs=1)

    # Освобождаем VRAM
    del extract_fn, whisper_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    dur_trainval = get_durations(paths_trainval).reshape(-1, 1)
    dur_test     = get_durations(paths_test).reshape(-1, 1)
    X_trainval = np.hstack([X_embeds_trainval, letters_trainval, dur_trainval])
    X_test     = np.hstack([X_embeds_test,     letters_test,     dur_test])

    # --- Этап 2: 5-fold CV (CPU) ---
    with start_run(exp_id, group="03_pretrained_frozen"):

        mlflow_params = {
            "encoder":        model_id,
            "encoder_frozen": True,
            "aggregation":    "mean+std+max",
            "embed_dim":      embed_dim,
            "classifier":     "SVM RBF",
            "cv_folds":       config.CV_N_SPLITS,
            "class_weight":   "balanced",
        }
        if extra_notes:
            mlflow_params["note"] = extra_notes
        mlflow.log_params(mlflow_params)

        fold_results = []
        t0 = time.perf_counter()

        for paths_tr, paths_val, labels_tr, labels_val, letters_tr, letters_val, fold in \
                get_cv_folds(paths_trainval, labels_trainval, letters_trainval):

            print(f"\nФолд {fold+1}/{config.CV_N_SPLITS}")

            idx_tr  = [path_to_idx[p] for p in paths_tr]
            idx_val = [path_to_idx[p] for p in paths_val]
            X_tr  = np.hstack([
                X_embeds_trainval[idx_tr],
                letters_tr,
                get_durations(paths_tr).reshape(-1, 1),
            ])
            X_val = np.hstack([
                X_embeds_trainval[idx_val],
                letters_val,
                get_durations(paths_val).reshape(-1, 1),
            ])

            grid = GridSearchCV(pipeline, param_grid, cv=3,
                                scoring="f1_macro", refit=True, n_jobs=-1)
            grid.fit(X_tr, labels_tr)
            clf = grid.best_estimator_
            print(f"  best={grid.best_params_}")

            y_proba_val = clf.predict_proba(X_val)[:, config.CLASS_BAD]
            thr = find_optimal_threshold(labels_val, y_proba_val)
            metrics = evaluate(labels_val, y_proba_val, threshold=thr, verbose=True)
            fold_results.append(metrics)
            log_cv_fold(fold, f1_bad=metrics["f1_bad"], f1_macro=metrics["f1_macro"],
                        roc_auc=metrics["roc_auc"], threshold=thr)

        train_time_sec = time.perf_counter() - t0
        cv_agg = evaluate_cv_folds(fold_results)
        print_cv_summary(cv_agg)

        _run_id = mlflow.active_run().info.run_id

    df_folds = pd.DataFrame(fold_results)
    df_folds.index = [f"Fold {i+1}" for i in range(len(fold_results))]
    return {
        "cfg":            cfg,
        "df_folds":       df_folds[["accuracy", "f1_macro", "f1_bad", "roc_auc", "threshold"]],
        "X_trainval":     X_trainval,
        "X_test":         X_test,
        "cv_agg":         cv_agg,
        "train_time_sec": train_time_sec,
        "embed_dim":      embed_dim,
        "_run_id":        _run_id,
    }


In [6]:
models_to_run = (
    WHISPER_MODELS if MODEL_FILTER is None
    else [m for m in WHISPER_MODELS if m["model_id"] == MODEL_FILTER]
)

result_csv = exp_dir / "result.csv"
if result_csv.exists():
    result_csv.unlink()

all_results = []
for cfg in models_to_run:
    result = run_whisper_experiment(cfg, result_csv_exists=result_csv.exists())
    all_results.append(result)
    display(result["df_folds"])



Модель: openai/whisper-small


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Извлечение признаков (train+val)...
Извлечение признаков (test)...

Фолд 1/5
  best={'clf__C': 5.0, 'clf__gamma': 'scale'}
              precision    recall  f1-score   support

        good       0.90      0.82      0.86       319
         bad       0.68      0.81      0.74       153

    accuracy                           0.82       472
   macro avg       0.79      0.81      0.80       472
weighted avg       0.83      0.82      0.82       472

Threshold : 0.37
Accuracy  : 0.8157
F1-macro  : 0.7987
F1-bad    : 0.7403
ROC-AUC   : 0.8809

Фолд 2/5
  best={'clf__C': 2.0, 'clf__gamma': 'scale'}
              precision    recall  f1-score   support

        good       0.90      0.84      0.87       319
         bad       0.70      0.80      0.75       152

    accuracy                           0.83       471
   macro avg       0.80      0.82      0.81       471
weighted avg       0.84      0.83      0.83       471

Threshold : 0.37
Accuracy  : 0.8259
F1-macro  : 0.8077
F1-bad    : 0.7485


,accuracy,f1_macro,f1_bad,roc_auc,threshold
Fold 1,0.815678,0.798721,0.740299,0.880939,0.37
Fold 2,0.825902,0.807675,0.748466,0.889354,0.37
Fold 3,0.819533,0.801490,0.741641,0.894221,0.36
Fold 4,0.825902,0.800868,0.730263,0.886353,0.42
Fold 5,0.791932,0.780158,0.729282,0.881860,0.22



Модель: openai/whisper-medium


...

,accuracy,f1_macro,f1_bad,roc_auc,threshold
Fold 1,0.845339,0.816543,0.743860,0.890426,0.55
Fold 2,0.838641,0.816069,0.751634,0.892839,0.46
Fold 3,0.815287,0.798485,0.740299,0.890117,0.33
Fold 4,0.836518,0.815192,0.752412,0.895768,0.37
Fold 5,0.804671,0.787746,0.727811,0.886011,0.26



Модель: qymyz/whisper-russian-dysarthria


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Извлечение признаков (train+val)...
Извлечение признаков (test)...

Фолд 1/5
  best={'clf__C': 2.0, 'clf__gamma': 'scale'}
              precision    recall  f1-score   support

        good       0.89      0.84      0.86       319
         bad       0.70      0.78      0.74       153

    accuracy                           0.82       472
   macro avg       0.80      0.81      0.80       472
weighted avg       0.83      0.82      0.82       472

Threshold : 0.41
Accuracy  : 0.8220
F1-macro  : 0.8026
F1-bad    : 0.7407
ROC-AUC   : 0.8759

Фолд 2/5
  best={'clf__C': 5.0, 'clf__gamma': 'scale'}
              precision    recall  f1-score   support

        good       0.88      0.81      0.85       319
         bad       0.66      0.78      0.72       152

    accuracy                           0.80       471
   macro avg       0.77      0.79      0.78       471
weighted avg       0.81      0.80      0.80       471

Threshold : 0.36
Accuracy  : 0.8004
F1-macro  : 0.7808
F1-bad    : 0.7152


,accuracy,f1_macro,f1_bad,roc_auc,threshold
Fold 1,0.822034,0.802628,0.740741,0.875909,0.41
Fold 2,0.800425,0.780778,0.715152,0.868710,0.36
Fold 3,0.828025,0.807440,0.744479,0.881022,0.41
Fold 4,0.796178,0.783257,0.730337,0.883456,0.25
Fold 5,0.813163,0.790472,0.721519,0.869970,0.31


In [7]:
for result in all_results:
    cfg            = result["cfg"]
    X_trainval     = result["X_trainval"]
    X_test         = result["X_test"]
    cv_agg         = result["cv_agg"]
    train_time_sec = result["train_time_sec"]
    embed_dim      = result["embed_dim"]
    _run_id        = result["_run_id"]

    exp_id      = cfg["exp_id"]
    exp_name    = cfg["exp_name"]
    model_label = cfg["model_label"]
    extra_notes = cfg.get("notes", "")

    print(f"\n{'='*60}\nФинальная модель: {cfg['model_id']}\n{'='*60}")

    with mlflow.start_run(run_id=_run_id):
        print("\nФинальная модель на train+val...")
        final_grid = GridSearchCV(pipeline, param_grid, cv=5,
                                  scoring="f1_macro", refit=True, n_jobs=-1)
        final_grid.fit(X_trainval, labels_trainval)
        mlflow.log_params({f"best_{k}": v for k, v in final_grid.best_params_.items()})

        cv_threshold = cv_agg["threshold_mean"]
        y_proba_test = final_grid.best_estimator_.predict_proba(X_test)[:, config.CLASS_BAD]
        test_metrics = evaluate(labels_test, y_proba_test, threshold=cv_threshold, verbose=True)
        pd.DataFrame({
            "path":    paths_test,
            "y_true":  labels_test,
            "y_pred":  (y_proba_test >= cv_threshold).astype(int),
            "y_proba": y_proba_test,
        }).to_csv(exp_dir / f"test_predictions_{exp_id}.csv", index=False)

        notes_str = f"5-fold CV + holdout | best={final_grid.best_params_} | thr={cv_threshold:.2f}"
        if extra_notes:
            notes_str = f"{extra_notes} | {notes_str}"

        save_result_csv(
            exp_dir=exp_dir, experiment_id=exp_id,
            experiment_name=exp_name,
            model=model_label,
            accuracy=test_metrics["accuracy"],     f1_macro=test_metrics["f1_macro"],
            f1_bad=test_metrics["f1_bad"],         roc_auc=test_metrics["roc_auc"],
            precision_bad=test_metrics["precision_bad"],
            recall_bad=test_metrics["recall_bad"],
            threshold=test_metrics["threshold"],
            cv_f1_bad_std=cv_agg["f1_bad_std"],
            cv_f1_macro_std=cv_agg["f1_macro_std"],
            cv_accuracy_std=cv_agg["accuracy_std"],
            cv_roc_auc_std=cv_agg["roc_auc_std"],
            cv_precision_bad_std=cv_agg["precision_bad_std"],
            cv_recall_bad_std=cv_agg["recall_bad_std"],
            cv_threshold_std=cv_agg["threshold_std"],
            embed_dim=embed_dim,
            embed_dim_note=f"Whisper encoder {embed_dim//3}-dim × 3 (mean+std+max)",
            notes=notes_str,
            train_time_sec=train_time_sec,
            append=result_csv.exists(),
        )
        print(f"  Результаты {exp_id} сохранены")

print("\nИтоговые результаты:")
display(pd.read_csv(result_csv))



Финальная модель: openai/whisper-small

Финальная модель на train+val...
              precision    recall  f1-score   support

        good       0.86      0.86      0.86       281
         bad       0.71      0.72      0.72       135

    accuracy                           0.81       416
   macro avg       0.79      0.79      0.79       416
weighted avg       0.82      0.81      0.82       416

Threshold : 0.35
Accuracy  : 0.8149
F1-macro  : 0.7893
F1-bad    : 0.7159
ROC-AUC   : 0.9018
  Результаты exp_whisper_svm сохранены

Финальная модель: openai/whisper-medium

Финальная модель на train+val...
              precision    recall  f1-score   support

        good       0.87      0.90      0.88       281
         bad       0.77      0.72      0.74       135

    accuracy                           0.84       416
   macro avg       0.82      0.81      0.81       416
weighted avg       0.84      0.84      0.84       416

Threshold : 0.39
Accuracy  : 0.8389
F1-macro  : 0.8130
F1-bad    

,experiment_id,experiment_name,model,accuracy,f1_macro,f1_bad,roc_auc,precision_bad,recall_bad,threshold,...,cv_accuracy_std,cv_f1_macro_std,cv_f1_bad_std,cv_roc_auc_std,cv_precision_bad_std,cv_recall_bad_std,cv_threshold_std,notes,num_params,train_time_sec
0,exp_whisper_svm,Whisper-small encoder + SVM,Whisper-small (frozen) + SVM RBF,0.814904,0.789306,0.715867,0.901753,0.713235,0.718519,0.348,...,0.012552,0.009302,0.007266,0.004904,0.032188,0.042209,0.067350,"5-fold CV + holdout | best={'clf__C': 1.0, 'cl...",NaN,4930.370682
1,exp_whisper_medium_svm,Whisper-medium encoder + SVM,Whisper-medium (frozen) + SVM RBF,0.838942,0.812979,0.743295,0.904732,0.769841,0.718519,0.394,...,0.025291,0.018998,0.014419,0.006212,0.072675,0.075766,0.137782,"5-fold CV + holdout | best={'clf__C': 1.0, 'cl...",NaN,6473.332642
2,exp_whisper_large_svm,Whisper-large encoder + SVM,Whisper-large (frozen) + SVM RBF,0.848558,0.821090,0.750988,0.902728,0.805085,0.703704,0.394,...,0.015442,0.011691,0.008963,0.003228,0.050082,0.043269,0.101311,"5-fold CV + holdout | best={'clf__C': 1.0, 'cl...",NaN,8040.191184
3,exp_whisper_dysarthria_svm,Whisper (Russian dysarthria finetune) + SVM,qymyz/whisper-russian-dysarthria (frozen) + SV...,0.810096,0.781278,0.701887,0.879794,0.715385,0.688889,0.348,...,0.012191,0.010506,0.011104,0.005834,0.028807,0.036469,0.061449,Whisper fine-tuned on Russian dysarthria speec...,NaN,3278.137874
